In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Recuperar e salvar resultados de jobs

<details>
<summary><b>Versões de pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
qiskit-ibm-runtime~=0.40.1
```
</details>
Os fluxos de trabalho quânticos frequentemente levam um tempo para serem concluídos e podem ser executados ao longo de muitas sessões. Reiniciar o kernel Python significa que você perderá todos os resultados armazenados na memória. Para evitar a perda de dados, você pode salvar resultados em arquivo e recuperar resultados de jobs anteriores do IBM Quantum&reg; para que sua próxima sessão possa continuar de onde parou.
## Recuperar resultados de jobs do IBM Quantum
O IBM Quantum armazena automaticamente os resultados de todos os jobs para que você possa recuperá-los em uma data posterior. Use esse recurso para continuar programas quânticos entre reinicializações de kernel e revisar resultados anteriores. Você pode obter o ID de um job programaticamente pelo método `job_id`, ou pode ver todos os jobs enviados e seus IDs na [página de Cargas de Trabalho](https://quantum.cloud.ibm.com/workloads).

Para encontrar um job programaticamente, use o método [`QiskitRuntimeService.jobs`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/qiskit-runtime-service#jobs). Por padrão, esse método retorna os jobs enviados mais recentemente, mas você também pode filtrar jobs por nome do Backend, data de criação e mais. A célula a seguir encontra todos os jobs enviados nos últimos três meses. O argumento `created_after` deve ser um objeto [`datetime.datetime`](https://docs.python.org/3.8/library/datetime.html#datetime.datetime).

In [1]:
import datetime
from qiskit_ibm_runtime import QiskitRuntimeService

three_months_ago = datetime.datetime.now() - datetime.timedelta(days=90)

service = QiskitRuntimeService()
jobs_in_last_three_months = service.jobs(created_after=three_months_ago)
jobs_in_last_three_months[:3]  # show first three jobs

[<RuntimeJobV2('d2n1d2cg59ks73c6vu3g', 'sampler')>,
 <RuntimeJobV2('d2n18dehb60s73cvalg0', 'sampler')>,
 <RuntimeJobV2('d2ff2cms6qcs738f4lb0', 'sampler')>]

You can also select by backend, job state, session, and more. For more information, see [`QiskitRuntimeService.jobs`](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#jobs) in the API documentation.

Once you have the job ID, use the [`QiskitRuntimeService.job`](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#job) method to retrieve it.

In [2]:
# Get ID of most recent successful job for demonstration.
# This will not work if you've never successfully run a job.
successful_job = next(
    j for j in service.jobs(limit=1000) if j.status() == "DONE"
)
job_id = successful_job.job_id()
print(job_id)

d2n1d2cg59ks73c6vu3g


In [3]:
retrieved_job = service.job(job_id)
retrieved_job.result()

PrimitiveResult([SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c

<CodeAssistantAdmonition
  tagLine="Always forgetting how to retrieve a job result? Try this prompt with Qiskit Code Assistant:"
  prompts={[
    "# Retrieve job JOB_ID from IBM Runtime and print the result"
  ]}
/>

## Save results to disk

You may also want to save results to disk. To do this, use Python's built-in JSON library with encoders from Qiskit Runtime.

In [4]:
import json
from qiskit_ibm_runtime import RuntimeEncoder

with open("result.json", "w") as file:
    json.dump(retrieved_job.result(), file, cls=RuntimeEncoder)

You can then load this array from disk in a separate kernel.

In [5]:
from qiskit_ibm_runtime import RuntimeDecoder

with open("result.json", "r") as file:
    result = json.load(file, cls=RuntimeDecoder)

result

PrimitiveResult([SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(c